In [ ]:
%run /home/mls07/speculative-decoding/notebooks/model.ipynb

import random
import numpy as np

In [2]:
OUTPUT_DIR = "/home/mls07/data"
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open("/data/train/prefix-train.txt", "r", encoding="utf-8") as file:
    data = file.read().split('\n\n')
    print(len(data))

650654


In [3]:
SEQUENCE_LEN = 128
GEN_LEN = 256
N_SAMPLES = 3000
TOP_K = 50
BATCH_SIZE = 8

In [4]:
def sample_token(token_ids):
    text = token_ids['input_ids'][0].tolist()
    n = len(text)
    if n < SEQUENCE_LEN:
        return None
    max_start = n - SEQUENCE_LEN
    start_idx = random.randint(0, max_start)
    return text[start_idx : start_idx + SEQUENCE_LEN]

In [5]:
def get_batch():
    text = []
    while len(text) < BATCH_SIZE:
        row = random.choice(data)
        token_ids = tokenizer(row, add_special_tokens=False, return_tensors="pt")
        p = sample_token(token_ids)
        if p is not None:
            text.append(p)
    return torch.tensor(text) 

In [6]:
@torch.no_grad()
def forward_teacher(data):
    input_ids = data.to(teacher.device)
    attention_mask = torch.ones_like(input_ids)
    B = input_ids.shape[0]
    result = torch.zeros(B, dtype=torch.bool, device=teacher.device) 
    all_topk_vals = []   
    all_topk_idx = []
    all_gen_tokens = []  
    outputs = teacher.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=GEN_LEN,
        do_sample=True,
        temperature=1.0,
        top_k=0,
        top_p=1.0,
        repetition_penalty=1.0,
        output_logits=True,
        return_dict_in_generate=True,
    )

    generated_ids = outputs.sequences          # [B, 128 + 256]
    output_logits = outputs.logits                # tuple of GEN_LEN, each [B, vocab_size]
    topk_vals = []
    topk_idx = []
    
    for logits in output_logits:
        vals, idx = torch.topk(logits, TOP_K, dim=-1)   # [B, TOP_K]
        topk_vals.append(vals.to(torch.float16).cpu())
        topk_idx.append(idx.to(torch.int32).cpu())
    
    return {
        "prefix_ids": input_ids.cpu(),                              # [B, SEQ_LEN]
        "generated_ids": generated_ids.cpu(),                       # [B, SEQ_LEN + GEN_LEN]
        "topk_logits": torch.stack(topk_vals, dim=1),      # [B, GEN_LEN, TOP_K]
        "topk_indices": torch.stack(topk_idx, dim=1),      # [B, GEN_LEN, TOP_K]
    }

In [7]:
i = 0
while i < N_SAMPLES:
    path = os.path.join(OUTPUT_DIR, f"batch_{i:06d}.pt")
    
    if os.path.exists(path):
        print(f"Vec postoji {path}")
        i += BATCH_SIZE
        continue
    
    batch = get_batch()
    result = forward_teacher(batch)
    torch.save(result, path)
    i += BATCH_SIZE
    print(f"[{i}/{N_SAMPLES}] saved {path}")
print("Done")

Vec postoji /home/mls07/data/batch_000000.pt
Vec postoji /home/mls07/data/batch_000008.pt
Vec postoji /home/mls07/data/batch_000016.pt
Vec postoji /home/mls07/data/batch_000024.pt
Vec postoji /home/mls07/data/batch_000032.pt
Vec postoji /home/mls07/data/batch_000040.pt
Vec postoji /home/mls07/data/batch_000048.pt
Vec postoji /home/mls07/data/batch_000056.pt
Vec postoji /home/mls07/data/batch_000064.pt
Vec postoji /home/mls07/data/batch_000072.pt
Vec postoji /home/mls07/data/batch_000080.pt
Vec postoji /home/mls07/data/batch_000088.pt
Vec postoji /home/mls07/data/batch_000096.pt
Vec postoji /home/mls07/data/batch_000104.pt
Vec postoji /home/mls07/data/batch_000112.pt
Vec postoji /home/mls07/data/batch_000120.pt
Vec postoji /home/mls07/data/batch_000128.pt
Vec postoji /home/mls07/data/batch_000136.pt
Vec postoji /home/mls07/data/batch_000144.pt
Vec postoji /home/mls07/data/batch_000152.pt
Vec postoji /home/mls07/data/batch_000160.pt
Vec postoji /home/mls07/data/batch_000168.pt
Vec postoj

In [8]:
print(teacher.generation_config.to_dict())

{'max_length': None, 'max_new_tokens': 2048, 'min_length': None, 'min_new_tokens': None, 'early_stopping': None, 'max_time': None, 'stop_strings': None, 'do_sample': False, 'num_beams': None, 'use_mtp': None, 'use_cache': None, 'cache_implementation': None, 'cache_config': None, 'max_cache_len': None, 'temperature': None, 'top_k': None, 'top_p': None, 'min_p': None, 'top_h': None, 'typical_p': None, 'epsilon_cutoff': None, 'eta_cutoff': None, 'repetition_penalty': None, 'encoder_repetition_penalty': None, 'length_penalty': None, 'no_repeat_ngram_size': None, 'bad_words_ids': None, 'renormalize_logits': None, 'forced_bos_token_id': None, 'forced_eos_token_id': None, 'remove_invalid_values': None, 'exponential_decay_length_penalty': None, 'suppress_tokens': None, 'begin_suppress_tokens': None, 'sequence_bias': None, 'token_healing': None, 'guidance_scale': None, 'watermarking_config': None, 'num_return_sequences': None, 'output_attentions': None, 'output_hidden_states': None, 'output_sco